# 03 · Statistics: what is an objective worth?

Logistic regression of Blue win on 15-minute state, gold in thousands.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("LOLWP_DATA_DIR", ROOT / "data"))
ART = Path(os.environ.get("LOLWP_ARTIFACTS_DIR", ROOT / "artifacts"))
sns.set_theme(style="whitegrid")
import statsmodels.formula.api as smf
state = pd.read_parquet(DATA / "state.parquet").assign(gold_k=lambda d: d.gold_diff / 1000)
fit = smf.logit("blue_win ~ gold_k + dragons_diff + towers_diff + grubs_diff + kills_diff",
                data=state).fit(disp=0)
fit.summary()

## Odds ratios with 95% intervals

In [ ]:
ci = np.exp(fit.conf_int()).drop("Intercept"); orr = np.exp(fit.params).drop("Intercept")
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(orr, orr.index, xerr=[orr - ci[0], ci[1] - orr], fmt="o"); ax.axvline(1, color="grey", ls="--")
ax.set_xscale("log"); ax.set_xlabel("Odds ratio (per unit, holding the others fixed)")
plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _What is one extra dragon, tower, or 1k gold worth in odds, holding the rest fixed?_

## Is one dragon worth the same as one tower?

In [ ]:
fit.t_test("dragons_diff = towers_diff")

**Interpretation (write 3–5 sentences after running on the real data):** _Can we reject equal value? Which is worth more, and by how much?_

## Caveat: association, not causation

These coefficients describe how 15-minute states relate to outcomes in this sample. Teams that take dragons may differ in ways the model doesn't see (draft, skill, map state), so an odds ratio is not the effect of forcing a dragon.